# Prepare non-overlapping NN-duel forecasts with four enemy-missile slots

Run this notebook before **18_enemy_missile_nn_duel_skill_forecasting.ipynb**.
It builds a new dataset from notebook 11's complete two-view corpus. The original
recordings already contain four friendly-missile slots for each aircraft, so no
new simulator collection is required. Opposing friendly missiles become this
observer's enemy missiles after a same-timestamp change of reference position and
velocity. Both observation streams must be available during preparation.

Each model frame contains **115 features** for the 33-skill catalogue:
**51 original observations + 3 causal accelerations + 28 enemy-missile values +
33 own-skill indicators**. Channels 54–81 are four consecutive slots, each with
relative North/West/Up position (3), relative North/West/Up velocity (3), and range
(1). Own-skill indicators start at channel 82. The existing four friendly-missile
slots remain among the original observation channels.

The new slots preserve the order of the opponent's four live friendly slots.
All-seven-zero source slots remain absent and zero before standardisation. This
matches the upstream compact interface's absence convention; there is no additional
presence bit. Range is recalculated relative to the observer. This is idealised
same-time simulator missile information: live forecasting requires equivalent
enemy-missile tracks. No opposing pilot decisions or future observations enter
these features. Skills remain opponent selections at exact +1/+2/+5 seconds.

Training uses the same **disjoint 56-raw-frame histories across both aircraft views**
as notebook 14, alternating views at six-second origins. Scenario splits, target
masks and dense test/validation windows match notebook 14 for the same source and
seed. Numeric scaling is fitted only on retained training histories; the 33 own-skill
indicators remain binary. The new contract prevents loading old 87-channel caches.

**Run All downloads required Git LFS payloads and prepares data; it does not collect
flights or train models.** Notebook 14 need not run first. Install `.[ml]` and an
IPython/Jupyter kernel. The archived raw source is about 10.3 GB; the separate
115-channel memory map is about **18.1 GiB**, plus indices and reports. Allow at least
35 GB for download and preparation, with additional space for existing arrays and
trained models. Data is streamed without expanding overlapping window tensors.
Every executable cell is timed. Data, receipts and runs use a separate artifact root.


In [ ]:
%%time
from pathlib import Path
import os
import sys

root_override = os.environ.get("BVR_NN_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in candidates
             if (p / "bvr_behavior_prediction/nn_duel_skill_forecasting.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set BVR_NN_ROOT to the full Trajectory_Classification checkout path.")
sys.path.insert(0, str(ROOT))
from bvr_behavior_prediction.notebook_runtime import require_pyarrow
require_pyarrow()  # Report the kernel environment and native-library repair steps.
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from bvr_behavior_prediction import nn_duel_skill_forecasting as duel_data
from bvr_behavior_prediction.paired_skill_forecasting import (
    HORIZONS, HISTORY_STEPS, RAW_HISTORY_STEPS, SPLITS, read_json, write_json, sha256,
)
EXPERIMENT_ROOT = ROOT / "artifacts/nn_duel_enemy_missile_forecasting"
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

SOURCE_OVERRIDE = os.environ.get("BVR_NN_FORECAST_SOURCE")  # Full path, or repository-relative.
SOURCE = duel_data.discover_corpus(ROOT, SOURCE_OVERRIDE)
SOURCE_MANIFEST = read_json(SOURCE / "manifest.json")
SOURCE_DIGEST = sha256(SOURCE / "manifest.json")
PREPARATION_DIGEST = sha256(duel_data.__file__)
FEATURE_HELPERS_DIGEST = sha256(ROOT / "bvr_behavior_prediction/paired_skill_forecasting.py")
DATA_DEFAULT = (EXPERIMENT_ROOT / "data" /
                f"{SOURCE.name}_{SOURCE_DIGEST[:12]}_{PREPARATION_DIGEST[:8]}_{FEATURE_HELPERS_DIGEST[:8]}")
DATA = Path(os.environ.get("BVR_NN_ENEMY_MISSILE_DATA", str(DATA_DEFAULT))).expanduser().resolve()
DOWNLOAD_LFS = os.environ.get("BVR_NN_FORECAST_DOWNLOAD", "1") == "1"
GIT_EXECUTABLE = None  # e.g. r"C:\Program Files\Git\cmd\git.exe" on Windows.
SPLIT_SEED = 1729
print("Source:", SOURCE)
print("Complete:", SOURCE_MANIFEST["complete"], "| physical flights:",
      SOURCE_MANIFEST["physical_flights"], "| perspectives:", SOURCE_MANIFEST["perspective_episodes"])
print("Source manifest SHA-256:", SOURCE_DIGEST)
print("Prepared output:", DATA)


## Alignment and forecasting contract

* Histories contain **51 chronological frames at 10 Hz**, spanning five seconds.
  Five additional past frames support backward 0.5-second acceleration. The first
  forecast origin is 6 seconds. Training uses six-second slots across both views,
  alternating the observer. Every training raw-history interval is disjoint from
  every other retained interval in the same flight. Test/validation use one-second
  stride. No input history crosses a flight.
* For observer A, its own A decision comes from **B's `target_decisions` file**.
  Actual aircraft IDs and episode keys are checked before alignment. A decision
  at tick 60 applies at ticks 60–69; the decision at 70 cannot affect earlier inputs.
  Missing own decision intervals fail preparation rather than being filled indefinitely.
  This is the pilot's selected skill known when issuing guidance at the frame timestamp.
* Future opponent labels require an **exact recorded decision**. Missing +1/+2/+5
  labels are masked independently with `-1`; there is no extrapolation beyond termination.
  Opponent decisions are labels only. Pilot/checkpoint IDs and outcomes are metadata.
* **40% train / 30% test / 30% validation**, grouped by `scenario_group`, keep
  both perspectives of every physical flight together. Seeded range strata preserve
  starting-range balance while balancing per-skill/horizon episode and window support.
  Continuous NN-duel ranges require strata in place of 09b's discrete range quotas.
  Splits are balanced on the original eligible candidate set before training thinning,
  matching notebook 12. Rare skills may lose support after thinning; inspect the
  retained per-class/horizon support and sampling audits.
* Scaling is fitted on unique frames used by **retained training histories only**. The
  appended skill indicators stay binary (mean 0, scale 1). No future labels or
  validation frames enter normalization. Earlier corpus files and 09b caches are preserved.

The enemy-missile reconstruction uses only matching `(episode_id, step, time_s)`
rows. For A observing B, `missile_position_relative_to_A = missile_position_relative_to_B
+ B_position_relative_to_A`, with the same identity for velocity. Position channels
use scales `[50000, 50000, 10000]` metres; velocity channels use 1200 m/s, so the
relative-aircraft velocity channels (scaled by 600 m/s) contribute half their value.
Range is computed after restoring physical position units, then divided by 50000 m.
Empty source slots are masked before they can become false missiles at B's position.


In [ ]:
%%time
if DOWNLOAD_LFS:
    duel_data.download_corpus(ROOT, SOURCE, git_executable=GIT_EXECUTABLE)

analysis = duel_data.analyze_corpus(ROOT, SOURCE)
support = duel_data.skill_support_table(analysis)
print(f"Surveyed {len(analysis['episodes']):,} perspectives before assigning splits.")
display(analysis["episodes"].groupby(["observer_id", "family"]).agg(
    perspectives=("episode_id", "size"), mean_duration_s=("duration_s", "mean")))
with pd.option_context("display.max_rows", None):
    display(support)
print("Zero-window catalogue labels:", sorted(set(analysis["classes"]) -
      set(support.loc[support.windows.gt(0), "skill"])))


In [ ]:
%%time
if (DATA / "manifest.json").exists():
    manifest = read_json(DATA / "manifest.json")
    if SOURCE_DIGEST not in manifest["source_manifests"].values():
        raise ValueError("The existing dataset belongs to another source; choose a new DATA path.")
    if manifest["source_code_sha256"] != PREPARATION_DIGEST:
        raise ValueError("Preparation code changed; choose a fresh DATA path.")
    if manifest["feature_helpers_sha256"] != FEATURE_HELPERS_DIGEST:
        raise ValueError("Feature helper code changed; choose a fresh DATA path.")
    if manifest["split_seed"] != SPLIT_SEED:
        raise ValueError("Split seed changed; choose a fresh DATA path.")
    print("Reusing the verified prepared dataset:", DATA)
else:
    manifest = duel_data.prepare_dataset(ROOT, DATA, source=SOURCE,
                                         split_seed=SPLIT_SEED, analysis=analysis,
                                         nonoverlapping_train=True, enemy_missiles=True)
data = duel_data.EnemyMissileNNDuelData(DATA)  # Verify stored file hashes and the input/target contract.
assert data.features.shape[1] == 82 + len(data.classes)
assert data.manifest["features"] == manifest["features"]
display(pd.DataFrame(manifest["counts"]).T)
display(pd.Series({"frames": len(data.features), "features_per_frame": data.features.shape[1],
                   "feature_array_GiB": data.features.nbytes / 2**30,
                   "history_steps": HISTORY_STEPS, "horizons_s": HORIZONS}))

# The loader verifies disjoint raw intervals across both views, not just per observer.
display(pd.Series(duel_data.nonoverlap_audit(data.samples)))
sampling = pd.read_csv(DATA / "window_sampling_audit.csv")
display(sampling.groupby("split")[["eligible_windows", "retained_windows", "dropped_windows"]].sum())

# Show physical-flight intervals across both observer views on the same time axis.
training_intervals = data.samples.loc[data.samples.split.eq("train"),
    ["episode_id", "observer_id", "step"]].sort_values(["episode_id", "step"])
training_intervals = training_intervals.assign(
    raw_start_tick=training_intervals.step - RAW_HISTORY_STEPS + 1,
    model_start_tick=training_intervals.step - HISTORY_STEPS + 1,
    raw_end_tick=training_intervals.step)
display(training_intervals.head(12))

missile_audit = pd.read_csv(DATA / "enemy_missile_audit.csv")
display(missile_audit.groupby(["split", "observer_id"])[
    ["raw_frames", "frames_with_enemy_missiles", *[f"slot_{s}_nonempty_frames" for s in range(1, 5)]]].sum())
print("Enemy-missile channels:", manifest["features"][54:82])


## Inspect the split and the appended inputs

The support tables retain zero-support classes. A missing class in validation is
visible here and is not repaired by changing the held-out partition after training.
The example below checks a normalized history from a training sample: its appended
indicators still identify the observing aircraft's own skill at every frame.

`skill_support_before_split.csv` describes all eligible candidates;
`skill_support_by_split.csv` describes the actual retained labels after thinning.
`window_sampling_audit.csv` records eligible/retained/dropped windows per flight view,
and `nonoverlap_audit.json` records verified raw-history disjointness.

`enemy_missile_audit.csv` reports raw-frame slot occupancy per flight/view. Raw
absent slots contain zeros; standardised values depend on fitted training means.
The example below displays the four reconstructed enemy slots in metres and m/s
at the forecast origin, together with the unchanged own-skill and target checks.


In [ ]:
%%time
assignments = pd.read_csv(DATA / "split_assignments.csv")
assert assignments.groupby("scenario_group").split.nunique().eq(1).all()
assert assignments.groupby("episode_id").observer_id.nunique().eq(2).all()
assert assignments.groupby("episode_id").split.nunique().eq(1).all()
display(assignments.groupby(["split", "observer_id"]).size().unstack(fill_value=0))
split_support = pd.read_csv(DATA / "skill_support_by_split.csv")
with pd.option_context("display.max_rows", None):
    display(split_support.pivot(index=["horizon_s", "skill"], columns="split",
                                values=["episodes", "windows", "scenario_groups"]))
sample_id = int(data.indices["train"][0])
x, y = data.batch(np.asarray([sample_id]), "cpu")
own = x[0, :, 82:].numpy()
assert np.isin(own, [0, 1]).all() and np.all(own.sum(axis=1) == 1)
assert np.all(data.mean[82:] == 0) and np.all(data.std[82:] == 1)
sample = data.samples.iloc[sample_id]
display(sample)
display(pd.DataFrame({
    "time_s": np.arange(sample.step - HISTORY_STEPS + 1, sample.step + 1) * .1,
    "observer_current_skill": np.asarray(data.classes)[own.argmax(axis=1)]}))
display(pd.Series({f"opponent_at_+{h}s": data.classes[int(t)] if t >= 0 else "MASKED"
                   for h, t in zip(HORIZONS, y[0].numpy())}))

# Display the raw reconstructed values in physical units for this sample origin.
raw_enemy = data.features[int(data.anchors[sample_id]), 54:82].reshape(4, 7)
enemy_physical = raw_enemy * [50000, 50000, 10000, 1200, 1200, 1200, 50000]
display(pd.DataFrame(enemy_physical, index=[f"enemy_{s}" for s in range(1, 5)], columns=[
    "relative_n_m", "relative_w_m", "relative_u_m", "relative_vn_mps",
    "relative_vw_mps", "relative_vu_mps", "range_m"]))

# Written only after preparation and verification finish. Notebook 18 ingests this exact dataset.
write_json(EXPERIMENT_ROOT / "latest_dataset.json", {
    "directory": str(DATA), "manifest_sha256": sha256(DATA / "manifest.json"),
    "source": str(SOURCE), "source_manifest_sha256": SOURCE_DIGEST})
print("Ready for notebook 18:", DATA)
